In [ ]:
%pip install agentops langgraph langchain-openai langchain-tavily python-dotenv

In [1]:

import os
# Impedir inicialização do tracing LangSmith ao importar o LangChain
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"
import re
import logging
from typing import TypedDict

import agentops
from agentops.sdk.decorators import operation, trace
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [3]:
# Carregar variáveis de ambiente
load_dotenv(r"C:\Users\eugen\Documents\ProjetoIA\.env", override=True)

# Desativar o envio de traces ao LangSmith (o AgentOps continua ativo)
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

for key in ("OPENAI_API_KEY", "AGENTOPS_API_KEY", "TAVILY_API_KEY"):
    if not os.getenv(key):
        raise ValueError(f"Variável ausente no .env: {key}")

# Evitar UnicodeEncodeError em mensagens do AgentOps no Windows/cp1252
class SafeAgentOpsLog(logging.Filter):
    def filter(self, record):
        record.msg = record.getMessage().encode("cp1252", "backslashreplace").decode("cp1252")
        record.args = ()
        return True

logging.getLogger("agentops").addFilter(SafeAgentOpsLog())
logging.basicConfig(level=logging.INFO)

In [4]:
# Evita a instrumentação OpenAI incompatível com alguns SDKs recentes.
# Os traces explícitos dos nós continuam ativos.
agentops.init(
    api_key=os.environ["AGENTOPS_API_KEY"],
    default_tags=["langgraph-agent"],
    auto_start_session=False,
    instrument_llm_calls=False,
    log_level="INFO",
    log_session_replay_url=True,
)

🖇 AgentOps: You're on the agentops free plan \U0001f914


In [5]:
class AgentState(TypedDict):
    query: str
    search_results: str
    summary: str
    final_answer: str

In [6]:
# Inicializar o modelo da OpenAI
model = ChatOpenAI(model="gpt-4o", temperature=0)

# Inicializar o Tavily Search
tavily_search = TavilySearch(max_results=3)

In [7]:
@operation(name="web_search")
def web_search(state: AgentState) -> AgentState:
    try:
        # Realizar a busca utilizando o Tavily
        search_results = tavily_search.invoke({"query": state["query"]})
        results = search_results.get("results", []) if isinstance(search_results, dict) else search_results

        if not isinstance(results, list) or not results:
            raise RuntimeError("O Tavily não retornou resultados.")

        # Preservar as URLs das fontes
        formatted_results = "\n\n".join(
            f"Título: {r.get('title', '')}\nURL: {r.get('url', '')}\nConteúdo: {r.get('content', '')}"
            if isinstance(r, dict) else str(r)
            for r in results
        )
        state["search_results"] = formatted_results
    except Exception:
        logging.exception("Erro ao realizar a busca com Tavily")
        raise
    return state

In [8]:
@operation(name="summarize_results")
def summarize_results(state: AgentState) -> AgentState:
    prompt = ChatPromptTemplate.from_template(
        "Resuma as seguintes informações em um parágrafo conciso em português. "
        "Não invente fatos.\n\nInformações: {search_results}"
    )
    chain = prompt | model
    summary = chain.invoke({"search_results": state["search_results"]}).content
    state["summary"] = summary
    return state

In [9]:
@operation(name="generate_answer")
def generate_answer(state: AgentState) -> AgentState:
    prompt = ChatPromptTemplate.from_template(
        "Com base na pergunta '{query}' e nas informações resumidas abaixo, "
        "forneça uma resposta detalhada em português, utilizando bullet points "
        "quando necessário. Não invente fatos ou nomes de cursos. "
        "Não escreva URLs: as fontes serão adicionadas pela aplicação.\n\n"
        "Resumo: {summary}\n\nResultados originais: {search_results}"
    )
    chain = prompt | model
    final_answer = chain.invoke({
        "query": state["query"],
        "summary": state["summary"],
        "search_results": state["search_results"],
    }).content

    # Não aceitar URLs geradas pelo modelo; usar apenas as recuperadas pelo Tavily.
    final_answer = re.sub(r"\[([^\]]+)\]\(https?://[^)]+\)", r"\1", final_answer)
    final_answer = re.sub(r"https?://[^\s)]+", "", final_answer)
    urls = list(dict.fromkeys(re.findall(r"(?m)^URL: (https?://\S+)$", state["search_results"])))
    if urls:
        final_answer += "\n\n### Fontes\n" + "\n".join(f"- {url}" for url in urls)

    state["final_answer"] = final_answer
    return state

In [10]:
# Construção do grafo
builder = StateGraph(AgentState)

builder.add_node("web_search", web_search)
builder.add_node("summarize_results", summarize_results)
builder.add_node("generate_answer", generate_answer)

builder.add_edge(START, "web_search")
builder.add_edge("web_search", "summarize_results")
builder.add_edge("summarize_results", "generate_answer")
builder.add_edge("generate_answer", END)

graph = builder.compile()

In [11]:
@trace(name="langgraph-workflow", tags=["langgraph-agent"])
def execute_workflow(query: str) -> str:
    initial_state = {
        "query": query,
        "search_results": "",
        "summary": "",
        "final_answer": "",
    }
    final_state = graph.invoke(initial_state)
    answer = final_state.get("final_answer")
    if not answer:
        raise RuntimeError("O nó generate_answer não produziu uma resposta.")
    return answer

In [12]:
query = "Quais são os cursos da Scoras Academy?"
answer = execute_workflow(query)
print("Resposta Final:")
display(Markdown(answer))

🖇 AgentOps: Session Replay for langgraph-workflow trace: https://app.agentops.ai/sessions?trace_id=ee9d6463e151836eacb6b8810c4da0ac
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
🖇 AgentOps: Session Replay for langgraph-workflow.session trace: https://app.agentops.ai/sessions?trace_id=ee9d6463e151836eacb6b8810c4da0ac


Resposta Final:


A Scoras Academy é uma instituição pioneira no Brasil, especializada na formação de agentes de inteligência artificial (IA). Sob a liderança de Anderson Amaral, a academia oferece uma variedade de cursos voltados tanto para iniciantes quanto para profissionais experientes, com foco na aplicação prática da IA em diversos setores. A seguir, estão listados os principais cursos oferecidos pela Scoras Academy:

- **Fundamentos de LLMs (Large Language Models):** Curso que aborda os conceitos básicos e avançados de modelos de linguagem de grande escala, essenciais para o desenvolvimento de soluções de IA.

- **Curso de Modelos de Linguagem Pequenos (SLMs):** Focado em modelos de linguagem menores, este curso é ideal para quem deseja entender as diferenças e aplicações específicas desses modelos.

- **Curso de RAG (Retrieval-Augmented Generation):** Explora técnicas de geração de conteúdo aumentadas por recuperação de informações, uma área emergente na IA.

- **Curso de LangGraph:** Introduz os participantes ao LangGraph, uma ferramenta para prototipagem rápida de soluções com LLMs, RAG e agentes de IA.

- **Desenvolvimento de Agentes de IA:** Curso abrangente que cobre desde a criação até a implementação de agentes de inteligência artificial.

- **Workflows Agentes:** Focado na criação e gestão de fluxos de trabalho inteligentes, utilizando agentes de IA para otimizar processos.

- **Roteamento de LLMs:** Curso que ensina como direcionar e gerenciar modelos de linguagem de forma eficiente.

- **Programação em Python para IA:** Oferece uma base sólida em Python, a linguagem de programação mais utilizada no desenvolvimento de soluções de IA.

- **Estratégia de Produto e Tomada de Decisão com IA:** Curso que combina inteligência artificial com estratégias de negócios para melhorar a tomada de decisões.

Além dos cursos, a Scoras Academy oferece:

- **Aprendizado Prático:** Os cursos são modulados para aplicação imediata em projetos reais, com código pronto para adaptação e acesso a materiais exclusivos no GitHub.

- **Suporte Contínuo:** Inclui sessões de perguntas e respostas (Q&A), fóruns de discussão e feedback personalizado para auxiliar no progresso dos alunos.

- **Certificações Reconhecidas:** Ao concluir os cursos, os participantes recebem certificações que comprovam suas habilidades e conhecimentos adquiridos.

A Scoras Academy, com sede em Barueri, São Paulo, destaca-se por sua abordagem prática e adaptável, preparando indivíduos para a transformação digital através da inteligência artificial.

### Fontes
- https://hotmart.com/pt-br/marketplace/produtos/scoras-academy/V95277538C
- https://scorasacademy.com.br
- https://tavily.com/knowledge/company/57WjqyzWZIff